# Mask R-CNN을 룰베이스 마스크로 fine-tuning

**접근**: `data/raw2`에 룰베이스 `get_mask()`를 돌려 마스크를 뽑고, `classify_shape()` 결과가 폴더 라벨과 **일치하는 것만** "가짜 정답(pseudo-label)"으로 채택해서 Mask R-CNN을 fine-tuning한다. 배경/전경(텀블러) 2클래스로만 학습 — 형태(4종) 분류는 여전히 `classify_shape()`가 담당하고, Mask R-CNN은 세그멘테이션 역할만 한다 (CLAUDE.md 설계 원칙).

**왜 필터링이 필요한가**: 룰베이스 마스크를 전부 쓰면 이미 알려진 실패작(예: taper_step에서 몸통을 놓친 마스크)까지 정답으로 학습시켜서 모델을 오히려 망가뜨린다. `classify_shape()` 결과가 폴더 라벨과 맞은 것만 "그럭저럭 괜찮은 마스크"로 간주해 필터링한다 (완벽한 보장은 아니지만 합리적인 대리 신호).

**공정한 평가를 위한 필수 조건**: `maskrcnn_vs_rule_based.ipynb`에서 평가에 썼던 40장(seed=42)은 **학습 데이터 풀에서 반드시 제외**한다 — 안 그러면 그 사진들로 평가할 때 데이터 누수(leakage)가 생겨서 "좋아졌다"는 착시가 생길 수 있다.

**알려진 리스크** (`docs/troubleshooting.md`, 대화 기록 참고): 룰베이스 마스크는 Mask R-CNN이 COCO에서 배운 것보다 경계가 거칠어서, 오히려 마스크 품질(경계 정밀도)이 나빠질 수도 있다. 파인튜닝 후 같은 40장으로 다시 평가해서 실제로 좋아졌는지 반드시 확인한다.

In [1]:
import sys, os, glob, random, time
from collections import Counter

sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision.transforms.functional as TF
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

from rule_based.shape_classifier import get_mask, classify_shape, SHAPE_LABELS_KO

DATA_DIR = os.path.join("..", "data", "raw2")
CLASSES = ["straight", "taper_smooth", "taper_step", "mug"]
RANDOM_SEED = 42
SAMPLES_PER_CLASS = 10

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


def imread_unicode(path):
    data = np.fromfile(path, dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

device: cuda


## 1. 평가용 40장(seed=42) 재현 — 학습에서 반드시 제외

In [2]:
random.seed(RANDOM_SEED)
eval_paths = {}
eval_set = set()
for cls in CLASSES:
    files = sorted(glob.glob(os.path.join(DATA_DIR, cls, "*.jpg")))
    sampled = random.sample(files, min(SAMPLES_PER_CLASS, len(files)))
    eval_paths[cls] = sampled
    eval_set.update(sampled)
print(f"평가셋(학습 제외 대상): {len(eval_set)}장")

평가셋(학습 제외 대상): 40장


## 2. 룰베이스 마스크로 가짜 정답 만들기 (평가셋 제외, classify_shape 일치만 채택)

전체 데이터를 훑는 단계라 시간이 좀 걸린다 (400여 장 × get_mask()).

In [3]:
t0 = time.time()
pseudo_items = []  # (image_bgr, mask) 튜플
kept_per_class = Counter()
skipped_per_class = Counter()

for cls in CLASSES:
    files = sorted(glob.glob(os.path.join(DATA_DIR, cls, "*.jpg")))
    pool = [f for f in files if f not in eval_set]
    for path in pool:
        image = imread_unicode(path)
        mask = get_mask(image)
        if not mask.any():
            skipped_per_class[cls] += 1
            continue
        result = classify_shape(mask)
        if result["shape"] == cls:
            pseudo_items.append((image, mask))
            kept_per_class[cls] += 1
        else:
            skipped_per_class[cls] += 1

print(f"소요 시간: {time.time()-t0:.1f}s")
print(f"{'클래스':14s} {'채택':>6s} {'제외':>6s}")
for cls in CLASSES:
    print(f"{cls:14s} {kept_per_class[cls]:6d} {skipped_per_class[cls]:6d}")
print(f"총 학습 샘플: {len(pseudo_items)}장")

소요 시간: 729.1s
클래스                채택     제외
straight           81     35
taper_smooth       64     27
taper_step         33     66
mug                88     11
총 학습 샘플: 266장


## 3. Dataset / DataLoader

In [4]:
class PseudoLabelDataset(torch.utils.data.Dataset):
    """룰베이스 마스크를 정답 삼아 배경(0)/텀블러(1) 2클래스로 학습시키기 위한 Dataset.

    형태(4종) 라벨은 쓰지 않는다 — Mask R-CNN은 세그멘테이션만 담당(CLAUDE.md 설계).
    """

    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        image_bgr, mask = self.items[idx]
        image_rgb = image_bgr[:, :, ::-1]
        image = TF.to_tensor(np.ascontiguousarray(image_rgb))

        ys, xs = np.nonzero(mask)
        box = torch.tensor([[xs.min(), ys.min(), xs.max(), ys.max()]], dtype=torch.float32)
        label = torch.tensor([1], dtype=torch.int64)
        mask_t = torch.tensor((mask > 0).astype(np.uint8))[None, :, :]

        target = {
            "boxes": box,
            "labels": label,
            "masks": mask_t,
            "image_id": torch.tensor([idx]),
            "area": (box[:, 2] - box[:, 0]) * (box[:, 3] - box[:, 1]),
            "iscrowd": torch.zeros((1,), dtype=torch.int64),
        }
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = PseudoLabelDataset(pseudo_items)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn
)
print("학습 배치 수:", len(train_loader))

학습 배치 수: 133


## 4. 모델 준비 (사전학습 가중치 + 헤드를 2클래스로 교체)

In [5]:
model = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT)

num_classes = 2  # background + tumbler
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)

model.to(device)
print("모델 준비 완료 (박스/마스크 헤드를 2클래스로 교체)")

모델 준비 완료 (박스/마스크 헤드를 2클래스로 교체)


## 5. Fine-tuning

In [6]:
NUM_EPOCHS = 3

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.005, momentum=0.9, weight_decay=0.0005,
)

model.train()
t0 = time.time()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"epoch {epoch+1}/{NUM_EPOCHS}  loss={epoch_loss/len(train_loader):.4f}  "
          f"({time.time()-t0:.0f}s 누적)")

print(f"학습 완료: 총 {time.time()-t0:.0f}s")

epoch 1/3  loss=0.3760  (430s 누적)


epoch 2/3  loss=0.1931  (830s 누적)


epoch 3/3  loss=0.1396  (1197s 누적)
학습 완료: 총 1197s


In [7]:
os.makedirs(os.path.join("..", "checkpoints"), exist_ok=True)
ckpt_path = os.path.join("..", "checkpoints", "maskrcnn_finetuned_rulebase_pseudolabels.pth")
torch.save(model.state_dict(), ckpt_path)
print("저장:", ckpt_path)

저장: ..\checkpoints\maskrcnn_finetuned_rulebase_pseudolabels.pth


## 6. 같은 40장(seed=42)으로 파인튜닝 전/후 비교

`maskrcnn_vs_rule_based.ipynb`의 제로샷 결과(85%)와 직접 비교 가능 — 학습에 안 쓴 held-out 세트.

In [8]:
def get_mask_finetuned(image_bgr, model, device, score_threshold=0.5, mask_threshold=0.5):
    """파인튜닝된 모델은 라벨이 0/1(배경/텀블러)뿐이라 COCO 카테고리 필터링이 필요 없다."""
    image_rgb = image_bgr[:, :, ::-1]
    tensor = TF.to_tensor(np.ascontiguousarray(image_rgb)).to(device)
    with torch.no_grad():
        output = model([tensor])[0]

    scores = output["scores"].cpu().numpy()
    labels = output["labels"].cpu().numpy()
    masks = output["masks"].cpu().numpy()

    keep = (scores >= score_threshold) & (labels == 1)
    if not keep.any():
        return None
    scores, masks = scores[keep], masks[keep]
    best = int(np.argmax(scores))
    return (masks[best, 0] >= mask_threshold).astype(np.uint8) * 255


def safe_classify(mask):
    if mask is None or not mask.any():
        return {"shape": "미검출"}
    return classify_shape(mask)


model.eval()
results = []
for cls in CLASSES:
    for path in eval_paths[cls]:
        image = imread_unicode(path)
        mask = get_mask_finetuned(image, model, device)
        result = safe_classify(mask)
        results.append({"folder_class": cls, "file": os.path.basename(path),
                         "pred": result["shape"], "match": result["shape"] == cls})

header = f"{'클래스':14s} {'표본':>6s} {'파인튜닝 후':>12s}"
print(header)
print("-" * len(header))
for cls in CLASSES:
    cls_results = [r for r in results if r["folder_class"] == cls]
    matched = sum(r["match"] for r in cls_results)
    total = len(cls_results)
    print(f"{cls:14s} {total:6d} {matched}/{total} ({matched/total*100:4.0f}%)")

total = len(results)
matched = sum(r["match"] for r in results)
print("-" * len(header))
print(f"{'합계':14s} {total:6d} {matched}/{total} ({matched/total*100:4.0f}%)")
print()
print("참고: 제로샷(파인튜닝 전) 결과는 maskrcnn_vs_rule_based.ipynb 기준 85.0%(34/40)")

클래스                표본       파인튜닝 후
----------------------------------
straight           10 8/10 (  80%)
taper_smooth       10 6/10 (  60%)
taper_step         10 7/10 (  70%)
mug                10 10/10 ( 100%)
----------------------------------
합계                 40 31/40 (  78%)

참고: 제로샷(파인튜닝 전) 결과는 maskrcnn_vs_rule_based.ipynb 기준 85.0%(34/40)


## 7. 혼동 분포 (파인튜닝 후)

In [9]:
for cls in CLASSES:
    cls_results = [r for r in results if r["folder_class"] == cls]
    counts = Counter(r["pred"] for r in cls_results)
    breakdown = ", ".join(f"{SHAPE_LABELS_KO.get(k, k)}({k})={v}" for k, v in counts.most_common())
    print(f"{cls:14s}: {breakdown}")

straight      : 직선 원통형(straight)=8, 연속 테이퍼형(taper_smooth)=1, 단차 테이퍼형(taper_step)=1
taper_smooth  : 연속 테이퍼형(taper_smooth)=6, 머그형(mug)=2, 직선 원통형(straight)=2
taper_step    : 단차 테이퍼형(taper_step)=7, 머그형(mug)=1, 연속 테이퍼형(taper_smooth)=1, 직선 원통형(straight)=1
mug           : 머그형(mug)=10


**결론은 실행 결과(숫자)를 보고 판단할 것** — 사전에 의도한 결과를 가정하지 않는다. 파인튜닝 전(85%)보다 낮아지면 우려했던 대로 "룰베이스의 거친 마스크를 학습해서 오히려 나빠진 것"일 가능성이 크고, 비슷하거나 나아지면 도메인 적응 효과가 리스크보다 컸다는 뜻이다.